# Évaluation de la méthode d'estimation de temps de course par factorisation pour comparaison avec la méthode par convolution

In [1]:
%load_ext autoreload
%autoreload 2

In [19]:
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
import random

import h5py
import pandas as pd
import numpy as np
import torch

from points_methods.tensorial_product_estimator import estimate_tensorial_product

In [3]:
folder_path = Path.cwd() / "hdf5_data" / "1y_tables"
file_paths = list(folder_path.glob("*hdf5"))
lim_date = datetime(2023, 1, 1)
hidden_part = 0.15
file_paths = [f for f in file_paths if datetime.fromisoformat(f.name[:10]) < lim_date]

## Préparation des données

In [13]:
def prepare_data(fp):
    with h5py.File(fp, "r") as f:
        scores = f["scores"][:]  # type: ignore
        categories = f["categories"][:]  # type: ignore
        mask = f["mask"][:]  # type: ignore
        hash_t = f["hash"][:]  # type: ignore
    normalized_hash = hash_t % 10**6 / 10**6
    hash_comp = normalized_hash < hidden_part
    mask_visible = mask & (~hash_comp)
    mask_hidden = mask & (hash_comp)
    kept_cols = mask_visible.sum(axis=0) > 0
    scores = scores[:, kept_cols]
    categories = (categories[kept_cols,],)
    mask_visible = mask_visible[:, kept_cols]
    mask_hidden = mask_hidden[:, kept_cols]
    scores_df = pd.DataFrame(scores * mask_visible)
    participation_df = pd.DataFrame(mask_visible, dtype=int)
    return scores, categories, mask_visible, mask_hidden, scores_df, participation_df

## Calcul de l'erreur d'estimation moyenne

In [16]:
def median_estimation_error(score, estimation, mask):
    abs_err = torch.from_numpy(np.abs(estimation - score))
    return torch.median(abs_err[torch.from_numpy(mask)]), mask.sum().sum()

In [20]:
med_errors = list()
nb_estimations = list()
random.shuffle(file_paths)
for fp in tqdm(file_paths[:50]):
    scores, categories, mask_visible, mask_hidden, scores_df, participation_df = (
        prepare_data(fp)
    )
    competition_vector, competitor_vector = estimate_tensorial_product(
        scores_df, participation_df
    )
    scores_estimation = np.outer(competition_vector, competitor_vector)
    med_error, nb_estimation = median_estimation_error(
        scores, scores_estimation, mask_hidden
    )
    med_errors.append(med_error)
    nb_estimations.append(nb_estimation)

  0%|          | 0/50 [00:00<?, ?it/s]

100%|██████████| 50/50 [00:58<00:00,  1.18s/it]


In [22]:
np.array(med_errors).mean() * 90

np.float64(4.645117919173136)